In [21]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
df=pd.read_csv("netflix_titles.csv")

## Önişleme Tekrarı

In [22]:
df['director']=df['director'].fillna('Unknown')
df['cast']=df['cast'].fillna('Unknown')
df['country']=df['country'].fillna('Unknown')
df["date_added"] = pd.to_datetime(df["date_added"], dayfirst=True, errors="coerce")
df['yil']=df['date_added'].dt.year
df['ay']=df['date_added'].dt.month
df['sure']= df['duration'].str.extract('(\d+)').fillna(0).astype(int)
df['listed_in']=df['listed_in'].str.lower()
df['description']=df['description'].str.lower()
df["turler_list"]=df["listed_in"].str.split(", ")

<>:7: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:7: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\afay1\AppData\Local\Temp\ipykernel_19696\347398638.py:7: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  df['sure']= df['duration'].str.extract('(\d+)').fillna(0).astype(int)
C:\Users\afay1\AppData\Local\Temp\ipykernel_19696\347398638.py:7: Pandas4Warning: 'int' is not supported as a fill value for str dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  df['sure']= df['duration'].str.extract('(\d+)').fillna(0).astype(int)


## İçerik Tabanlı Öneri Sistemi

In [23]:
df["soup"] = (
    df["title"].fillna("") + " " +
    df["director"].fillna("") + " " +
    df["cast"].fillna("") + " " +
    df["country"].fillna("") + " " +
    df["rating"].fillna("") + " " +
    df["duration"].fillna("") + " " +
    df["listed_in"].fillna("") + " " +
    df["description"].fillna("")
)

In [24]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['soup'])

In [25]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [35]:
df = df.reset_index(drop=True)
indices = pd.Series(df.index, index=df["title"].str.lower()).drop_duplicates()

def oneri_getir(baslik, dosya_adi="oneriler.csv"):
    baslik_temiz = baslik.strip().lower()
    if baslik_temiz not in indices:
        print(f"'{baslik}' veri setinde bulunamadı.")
        return None

    idx = indices[baslik_temiz]
    en_benzer = pd.Series(cosine_sim[idx]).nlargest(11).iloc[1:]
    sonuc_df = df.loc[en_benzer.index, ["title", "listed_in", "release_year"]].copy()
    sonuc_df["benzerlik_skoru"] = en_benzer.values.round(3)

    print(sonuc_df)
    sonuc_df.to_csv(dosya_adi, index=False, encoding="utf-8-sig")
    return sonuc_df

In [36]:
# Örnek Kullanım:
oneriler = oneri_getir("Blood & Water")

                                       title  \
1514                            Diamond City   
1593                        Kings of Jo'Burg   
4475                                Shirkers   
3211  Kalushi: The Story of Solomon Mahlangu   
1905                            Cold Harbour   
1535                   How To Ruin Christmas   
7262                       Lady in the Water   
5854                         Frank and Cindy   
8316                     The Future of Water   
1955                  The School Nurse Files   

                                              listed_in  release_year  \
1514  crime tv shows, international tv shows, tv dramas          2019   
1593  crime tv shows, international tv shows, tv dramas          2020   
4475                                      documentaries          2018   
3211                       dramas, international movies          2016   
1905   action & adventure, dramas, international movies          2013   
1535     international tv shows, 